## Ensemble testes
Esse script implementa um ensemble por média entre dois modelos de Gradient Boosting:
* LightGBM (rodando em GPU, com categorias nativas)
* XGBoost (rodando em CPU, com árvores histogram)

## 1.Bibliotecas

In [1]:
import sys
import warnings
import os
import pandas as pd
import numpy as np
import joblib
import time
from pathlib import Path

# Modelagem e Métricas
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression

# Importações locais
from pathlib import Path
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.plot_metrica_class import *


import src.preprocess_utils_diab14 as new_utils
sys.modules['src.preprocess_utils_diab'] = new_utils


# from src.plot_metrica_class import * warnings.filterwarnings("ignore")
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")
start_inicial = time.time()

# Processo iniciado em: 14:59:00


## 2-DataLoad

In [2]:
# Definindo caminhos de pastas
BASE = Path.cwd().parent   
DATA_DIR = BASE / "data" / "raw"
DATA_MODELS = BASE / "models" 


# 1. Carregamento dos dados
# forma 1: usa todos os dados
dfo = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/train.csv")
df = dfo.drop(columns='id')
TARGET = "diagnosed_diabetes"
X_train = df.drop(columns=TARGET)
y_train = df[TARGET]

# forma 2: carrega os dados já separados.(aqui tem apenas 70%)...outros 30 será usado para validação.
# 1. Carregamento dos dados
#X_train = pd.read_csv(DATA_DIR / "X_train_raw.csv").reset_index(drop=True)
#y_train = pd.read_csv(DATA_DIR / "y_train_raw.csv").values.ravel()

# Dados de Teste (Kaggle)
df_test = pd.read_csv(DATA_DIR / "test.csv")
id_test = df_test["id"]
X_test_final = df_test.drop(columns='id')

# 2. Carregamento dos modelos pré-treinados
# Nota: Esses modelos geralmente são Pipelines que já incluem o preprocessador
pipe_XGB = joblib.load(DATA_MODELS / 'modelo_XGB_final_randsearch.roc_auc_v1.2.joblib') # AUC Score : 0.7264

modelo_XGB2_final_refine.roc_auc_v3.joblib
pipe_LGBM = joblib.load(DATA_MODELS / 'modelo_LGBM_final_refine.roc_auc_v1.2.joblib')    # AUC Score  0.7284
print("✅ Dados e modelos carregados com sucesso!")

✅ Dados e modelos carregados com sucesso!


In [3]:
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays vazios para guardar as probabilidades de cada modelo
oof_lgbm = np.zeros(len(X_train))
oof_xgb  = np.zeros(len(X_train))

print(f"🚀 Iniciando Cross-Validation ({N_SPLITS} folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    T_i_fold = time.time()
    # Fit nos dados de treino do fold
    pipe_LGBM.fit(X_train.iloc[train_idx], y_train[train_idx])
    pipe_XGB.fit(X_train.iloc[train_idx], y_train[train_idx])

    # Predict_proba nos dados de validação do treino fold
    oof_lgbm[val_idx] = pipe_LGBM.predict_proba(X_train.iloc[val_idx])[:, 1]
    oof_xgb[val_idx]  = pipe_XGB.predict_proba(X_train.iloc[val_idx])[:, 1]
    print(f"Fold {fold+1} concluído. Tempo: {(time.time()-T_i_fold):3.2f}s ")

# Cálculo do score base de cada um
print(f"\nScore Individual LGBM: {roc_auc_score(y_train, oof_lgbm):.4f}")
print(f"Score Individual XGB:  {roc_auc_score(y_train, oof_xgb):.4f}")
print(f"# Processo finalizado em: {time.strftime('%H:%M:%S')}")

#resultado anterior(pp antigo)
# # Processo iniciado em: 10:28:07
# 🚀 Iniciando Cross-Validation (5 folds)...
# Fold 1 concluído. Tempo: 49.12s 
# Fold 2 concluído. Tempo: 53.85s 
# Fold 3 concluído. Tempo: 54.28s 
# Fold 4 concluído. Tempo: 53.75s 
# Fold 5 concluído. Tempo: 55.75s 

# Score Individual LGBM: 0.7287
# Score Individual XGB:  0.7277
# # Processo finalizado em: 10:32:35

# Processo iniciado em: 14:59:37
🚀 Iniciando Cross-Validation (5 folds)...
Fold 1 concluído. Tempo: 108.93s 
Fold 2 concluído. Tempo: 121.69s 
Fold 3 concluído. Tempo: 141.54s 
Fold 4 concluído. Tempo: 119.41s 
Fold 5 concluído. Tempo: 105.30s 

Score Individual LGBM: 0.7296
Score Individual XGB:  0.7277
# Processo finalizado em: 15:09:34


In [4]:
thresholds = np.linspace(0.01, 0.99, 99)
oof_ensemble_base = 0.5 * oof_lgbm + 0.5 * oof_xgb # Média simples para teste

f1_scores = [f1_score(y_train, (oof_ensemble_base >= t).astype(int)) for t in thresholds]

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

# Alternativa: Índice de Youden (Equilíbrio entre Sensibilidade e Especificidade)
fpr, tpr, thresh_roc = roc_curve(y_train, oof_ensemble_base)
youden_threshold = thresh_roc[np.argmax(tpr - fpr)]

print(f"🎯 Melhor threshold (F1): {best_threshold:.3f} com F1 de {max(f1_scores):.4f}")
print(f"🎯 Threshold (Youden): {youden_threshold:.3f}")

# 🎯 Melhor threshold (F1): 0.400 com F1 de 0.7810
# 🎯 Threshold (Youden): 0.626

🎯 Melhor threshold (F1): 0.410 com F1 de 0.7812
🎯 Threshold (Youden): 0.626


In [5]:
# Treino final com 100% dos dados de treino
pipe_LGBM.fit(X_train, y_train)
pipe_XGB.fit(X_train, y_train)

# Predição na base de teste real
p_lgbm = pipe_LGBM.predict_proba(X_test_final)[:, 1]
p_xgb  = pipe_XGB.predict_proba(X_test_final)[:, 1]


In [6]:
# 1) Aplicando pesos de Blending (ajuste conforme peso manualmente)
p_ensemble_final = 0.72 * p_lgbm + 0.28 * p_xgb
# Aplicando o threshold escolhido 
y_pred_binario = (p_ensemble_final >= youden_threshold).astype(int)

# Gerando CSV
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_binario
})

submission.to_csv("submission_ensemble_v0_Blending_manual_nosplittrain_v1.2.csv", index=False)
print("✅ Arquivo de submissão salvo!")


✅ Arquivo de submissão salvo!


In [7]:
# 2) calculando os pesos via regressão logisca
# Criamos uma nova matriz onde as colunas são as previsões do LGBM e XGB

X_meta = np.column_stack([oof_lgbm, oof_xgb])
X_meta_test = np.column_stack([p_lgbm, p_xgb])

# Meta-modelo (Regressão Logística é o padrão para Stacking)
meta_model = LogisticRegression()
meta_model.fit(X_meta, y_train)

# Score do Stacking 
p_meta_oof = meta_model.predict_proba(X_meta)[:, 1]
print(f"🚀 ROC AUC do Stacking: {roc_auc_score(y_train, p_meta_oof):.4f}")


# 2. Usar o meta-modelo para prever as probabilidades finais
# O meta_model aplica os pesos e o intercepto automaticamente
p_stacking_final = meta_model.predict_proba(X_meta_test)[:, 1] 

fpr, tpr, thresholds_meta = roc_curve(y_train, p_meta_oof)
youden_threshold_stk = thresholds_meta[np.argmax(tpr - fpr)]

print(f"🎯 Novo Threshold Otimizado para Stacking: {youden_threshold_stk:.3f}")


y_pred_stacking = (p_stacking_final >= youden_threshold_stk).astype(int)


# 4. Criar o DataFrame de submissão
submission_stacking = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_stacking
})

submission_stacking.to_csv("submission_ensemble_v0_stacking_nosplittrain_v1.2.csv", index=False)
print("✅ Submissão via Stacking gerada com sucesso!")

# # Acessando os coeficientes (pesos) do meta-modelo
pesos = meta_model.coef_[0]
intercept = meta_model.intercept_[0]

# print(f"⚖️ Peso atribuído ao LGBM: {pesos[0]:.4f}")
# print(f"⚖️ Peso atribuído ao XGB:  {pesos[1]:.4f}")
# print(f"📌 Intercepto (Viés):     {intercept:.4f}")

# Para transformar em pesos percentuais (aproximados)
soma_pesos = np.abs(pesos).sum()
print(f"\nImportância Relativa (estimada):")
print(f"LGBM: {(np.abs(pesos[0])/soma_pesos)*100:.1f}%")
print(f"XGB:  {(np.abs(pesos[1])/soma_pesos)*100:.1f}%")

# 🚀 ROC AUC do Stacking: 0.7287
# 🎯 Novo Threshold Otimizado para Stacking: 0.646
# ✅ Submissão via Stacking gerada com sucesso!
#resultado anterior(pp antigo)
# Importância Relativa (estimada):
# LGBM: 50.6%
# XGB:  49.4%

🚀 ROC AUC do Stacking: 0.7297
🎯 Novo Threshold Otimizado para Stacking: 0.639
✅ Submissão via Stacking gerada com sucesso!

Importância Relativa (estimada):
LGBM: 87.7%
XGB:  12.3%


In [8]:
#) Otimização Numérica
from scipy.optimize import minimize

# Função que queremos MINIMIZAR (negativo do AUC)
def objective(weights):
    # Garante que as probabilidades combinadas fiquem entre 0 e 1
    p_ensemble = weights[0] * oof_lgbm + weights[1] * oof_xgb
    return -roc_auc_score(y_train, p_ensemble)

# Chute inicial (50/50)
v_inicial = [0.5, 0.5]
# Restrições: pesos entre 0 e 1; a soma deve ser 1.0
bounds = [(0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
res = minimize(objective, v_inicial, bounds=bounds, constraints=constraints)
w1, w2 = res.x
print(f"🏆 Pesos Ótimos: LGBM = {w1:.4f}, XGB = {w2:.4f}")
print(f"⭐ Melhor AUC possível via Blending: {-res.fun:.4f}")



#1) Aplicando pesos calculados no treino
p_oof_otmz = w1 * oof_lgbm + w2 * oof_xgb

fpr, tpr, thresh_roc = roc_curve(y_train, p_oof_otmz)
youden_threshold_otmz = thresh_roc[np.argmax(tpr - fpr)]

print(f"🎯 Threshold Otimizado (Youden): {youden_threshold_otmz:.4f}")

# 3) Aplicar os pesos nos dados de TESTE (Submissão)
p_test_otimizado = w1 * p_lgbm + w2 * p_xgb

# 4) Aplicar o threshold calibrado nas predições de teste
y_pred_otmz = (p_test_otimizado >= youden_threshold_otmz).astype(int)

# 5) Gerando CSV
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_otmz
})

submission.to_csv("submission_ensemble_v0_objet_nosplittrain_v1.2.csv", index=False)
print("✅ Submissão salva com sucesso!")
#resultado anterior(pp antigo)
# 🏆 Pesos Ótimos: LGBM = 0.5070, XGB = 0.4930
# ⭐ Melhor AUC possível via Blending: 0.7287
# 🎯 Threshold Otimizado (Youden): 0.6257

🏆 Pesos Ótimos: LGBM = 0.5364, XGB = 0.4636
⭐ Melhor AUC possível via Blending: 0.7293
🎯 Threshold Otimizado (Youden): 0.6256
✅ Submissão salva com sucesso!
